In [7]:
import os
import json
import pandas as pd
import numpy as np
import IPython
from thefuzz import process
from sklearn.metrics.pairwise import cosine_similarity

# ==================================================================================================
# === 1. PAGED BROWSING MECHANISM ==================================================================
# ==================================================================================================
def get_ui_song_list(songs, page_number):
    """
    Slices and displays a 10-track window from the active track pool with dynamic navigation UI.
    """
    start_idx = page_number * 10
    end_idx = min(start_idx + 10, len(songs))
    page_songs = songs[start_idx:end_idx]

    ui_buffer = [f"\n --- SONG LIST {page_number + 1} (Songs {start_idx + 1} to {end_idx}) ---"]
    for i, song in enumerate(page_songs, start_idx + 1):
        ui_buffer.append(f"[{i}] {song[0]} by {song[1]}")
    ui_buffer.append("----------------------------------------------")
    ui_buffer.append("🔘 [ ⬅️ Previous 10 ]  |  🔘 [ ➡️ Next 10 ]  |  🔘 [ Done ✅ ]")
    print("\n".join(ui_buffer))


# ==================================================================================================
# === 2. FLEXIBLE PREFIX-FUZZY SEARCH ENGINE =======================================================
# ==================================================================================================
def flexible_two_input_search(master_df) -> list:
    """
    Filters the catalog using a 2-character title prefix slice before passing 
    candidates to a fuzzy string matcher to rank search relevance.
    """
    print("\n🔍 --- NEW MATRIX SEARCH ---")
    print("(Press Enter on both fields or type 'escape' to return to browsing mode)")
    title_input = input("🎵 Enter Song Title (or press Enter to skip): ").strip().lower()
    
    if title_input == 'escape':
        return []
        
    artist_input = input("👤 Enter Artist Name (or press Enter to skip): ").strip().lower()
    
    if artist_input == 'escape':
        return []

    if not title_input and not artist_input:
        return []
    
    matched_df = master_df
    if title_input:
        title_prefix = title_input[:2]
        prefix_matches = master_df[master_df['name_prefix'] == title_prefix]
        
        if not prefix_matches.empty:
            matched_df = prefix_matches
            if artist_input:
                artist_subset = matched_df[matched_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]
                if not artist_subset.empty:
                    matched_df = artist_subset
        else:
            if artist_input:
                matched_df = master_df[master_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]
    elif artist_input:
        matched_df = master_df[master_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]

    if matched_df.empty:
        print("⚠️ No exact filters found. Scanning global text pool...")
        matched_df = master_df
    
    lookup_dictionary = {}
    for row in matched_df.itertuples():
        lookup_key = f"{str(row.name).lower()} - {str(row.clean_artists).lower()}"
        lookup_dictionary[lookup_key] = [row.name, row.clean_artists]

    choices_list = list(lookup_dictionary.keys())

    if title_input and artist_input:
        target_string = f"{title_input} - {artist_input}"
    elif title_input:
        target_string = title_input
    else: 
        target_string = artist_input

    raw_matches = process.extract(target_string, choices_list, limit=None)

    formatted_matches = []
    match_counter = 1
    print("\n 🎯 Matches Found:")

    for match_item, confidence_score in raw_matches:
        if confidence_score >= 70:
            matches = lookup_dictionary[match_item]
            formatted_matches.append(matches)
            print(f"[{match_counter}] {matches[0]} by {matches[1]} (Confidence: {int(confidence_score)})")
            match_counter += 1

    return formatted_matches


# ==================================================================================================
# === 3. CALCULATE TASTE VECTOR ===================================================================
# ==================================================================================================
def calculate_taste_vector(user_ratings, df):
    """
    Normalizes features, applies custom weights to star ratings, and generates
    a 9-dimensional vector representing the user's current session vibe profile.
    """
    features = [
        'danceability', 'energy', 'loudness', 'speechiness', 
        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
    ]
    
    working_df = df.copy()
    working_df['loudness'] = (working_df['loudness'] - (-60.0)) / (0.0 - (-60.0))
    working_df['tempo'] = (working_df['tempo'] - (0.0)) / (250.0 - (0.0))
    working_df[features] = np.clip(working_df[features], 0.0, 1.0)

    working_df['lookup_key'] = working_df.name.str.lower() + ' - ' + working_df.clean_artists.str.lower() 

    keys = working_df['lookup_key'].to_numpy()
    vectors = working_df[features].to_numpy(dtype=float)
    working_dict = dict(zip(keys, vectors))

    rating_weight_map = {5: 2.5, 4: 1.5, 3: 0.2, 2: -1.5, 1: -2.5}
    weighted_features_list = []
    total_weight = 0

    for song_name, artist_name, rating in user_ratings:
        search_string = song_name.lower() + ' - ' + artist_name.lower()
        if search_string not in working_dict:
            continue
        raw_vector = working_dict[search_string].copy()
        weighted_vector = raw_vector * rating_weight_map[rating]
        weighted_features_list.append(weighted_vector)
        total_weight += abs(rating_weight_map[rating])

    if not weighted_features_list or total_weight == 0:
        return np.zeros(9)
    
    taste_vector = np.sum(weighted_features_list, axis=0) / total_weight
    taste_vector = np.clip(taste_vector, 0.0, 1.0)

    return taste_vector


# ==================================================================================================
# === 4. GENERATE RECOMMENDATIONS (ROBUST MULTI-REGION PATTERN) ====================================
# ==================================================================================================
def generate_geo_recommendations(test_vector, master_df, exclude_tracks=None, target_region=None, n=1000000):
    """
    Performs vector similarity mappings across the feature database to return 
    the top 50 tracks matching the targeted taste vector. Handles multiple market targets seamlessly.
    """
    features = [
        'danceability', 'energy', 'loudness', 'speechiness', 
        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
    ]

    working_pool = master_df.copy()

    # --- UPDATED GEOGRAPHIC FILTER LAYER ---
    if target_region:  
        if isinstance(target_region, str):
            target_region = [target_region]
            
        if "Global" not in target_region:
            print(f"🎯 Filtering track pool down to markets: {', '.join(target_region)}...")
            
            # Replaced absolute evaluation '==' with collection containment '.isin()'
            region_mask = (
                (working_pool['region_1'].isin(target_region)) |
                (working_pool['region_2'].isin(target_region)) |
                (working_pool['region_3'].isin(target_region))
            )
            working_pool = working_pool[region_mask]

            if working_pool.empty:
                print(f"⚠️ Warning: No tracks found matching criteria. Falling back to full catalog.")
                working_pool = master_df.copy()

    print(f"🧮 Calculating similarities across {len(working_pool):,} available tracks...")

    if exclude_tracks:
        print(f"🛑 Filtering out {len(exclude_tracks)} historical/session tracks from recommendation pool...")
        pool_signatures = working_pool['name'].str.lower() + "|||" + working_pool['clean_artists'].str.lower()
        exclude_signatures = {f"{t[0].lower()}|||{t[1].lower()}" for t in exclude_tracks}

        working_pool = working_pool[~pool_signatures.isin(exclude_signatures)]

    sample_size = min(n, len(working_pool))
    print(f"🎬 Sampling {sample_size} tracks from master catalog for vector mapping...")

    random_sample = working_pool.sample(n=sample_size).copy()
    feature_matrix = random_sample[features].values.copy()

    feature_matrix[:, 2] = (feature_matrix[:, 2] - (-60.0)) / (0.0 - (-60.0))
    feature_matrix[:, 8] = (feature_matrix[:, 8] - 0.0) / (250.0 - 0.0)
    feature_matrix = np.clip(feature_matrix, 0.0, 1.0)
    
    test_vector = test_vector.reshape(1, -1)
    random_sample['cos_sim'] = cosine_similarity(test_vector, feature_matrix).flatten()
    random_sample.sort_values('cos_sim', ascending=False, inplace=True)

    print("\n✨ Generating Recommendations...")
    recommendations = []

    for row in random_sample.itertuples():
        if len(recommendations) >= 50:
            break
        if row.cos_sim >= 0.9999:
            continue
        recommendations.append([row.name, row.clean_artists])

    if len(recommendations) == 0:
        for row in random_sample.head(10).itertuples():
            recommendations.append([row.name, row.clean_artists])
    
    return recommendations


# ==================================================================================================
# === 5. HISTORICAL SESSION PERSISTENCE (JSON LAYER) ===============================================
# ==================================================================================================
def load_interacted_songs_from_disk(history_file_path):
    """
    Safe reads the user history file from disk and parses it back into a standard list.
    """
    if not history_file_path or not os.path.exists(history_file_path):
        return []
    with open(history_file_path, 'r') as file:
        past_songs_list = json.load(file)
    return past_songs_list


def save_interacted_songs_to_disk(current_session_selections, history_file_path):
    """
    Appends unique newly-interacted tracks onto the master log list and writes back to disk.
    """
    past_song_list = load_interacted_songs_from_disk(history_file_path)
    for track in current_session_selections: 
        track_info = [track[0], track[1]]
        if track_info not in past_song_list:
            past_song_list.append(track_info)
    with open(history_file_path, 'w') as file:
        json.dump(past_song_list, file, indent=4)


# ==================================================================================================
# === 6. SPLIT-SOURCE ONBOARDING INITIALIZATION ===================================================
# ==================================================================================================
def initialize_onboarding_tracks(master_df, history_file_path):
    """
    Handles startup configuration by routing returning profiles or fresh random catalogs.
    """
    if os.path.exists(history_file_path):
        past_songs_list = load_interacted_songs_from_disk(history_file_path)
    else: 
        past_songs_list = []

    if len(past_songs_list) > 0:
        print("👋 Welcome Back to Bearly Listening!")
        print("[1] Load my previously rated songs onto the landing page")
        print("[2] Give me a fresh, random selection of songs")

        input_choice = input("Press 1 or 2: ").strip()
        if input_choice == "1":
            print("🔄 Loading your historical tracks...")
            return past_songs_list
    
    print("🎲 Generating a fresh discovery catalog...")
    random_sample = master_df.sample(n=50)[['name', 'clean_artists']].values.tolist()
    return random_sample


# ==================================================================================================
# === 7. APPLICATION INTERACTION RUN LOOP =========================================================
# ==================================================================================================
def run_interaction_loop(master_songs_list, master_df, history_file_path):
    current_page = 0
    user_selections = []
    active_view_tracks = master_songs_list
    is_running = True
    loop_count = 0
    max_safe_steps = 1000
    current_mode = "browse"

    print("🎵 Welcome to the Bearly Listening Mood Matcher! 🎵")
    print("Tell me 3 or 4 songs that match your current vibe.")
    
    while is_running == True:
        loop_count += 1
        if loop_count > max_safe_steps:
            print("⚠️ Runaway loop guard triggered. Safely exiting session.")
            break

        IPython.display.clear_output(wait=True)

        if current_mode == 'browse':
            print("🎵 --- BASE ONBOARDING TRACK CATALOG ---")
            get_ui_song_list(active_view_tracks, current_page)
        elif current_mode == 'search':
            print("🔍 Active Search Results Matrix")
            get_ui_song_list(active_view_tracks, current_page)            
            print("--------------------------------------------------------")
            print("Options: ['back' to escape search mode] or ['done' to view final picks]")
        elif current_mode == 'recommendations':
            print("✨ --- YOUR PERSONALIZED RECOMMENDATIONS SPECTRUM ---")
            get_ui_song_list(active_view_tracks, current_page)
            print("--------------------------------------------------------")
            print("Options: ['next'/'prev' to flip pages] | ['exit' to close application]")

        print(f"🛒 Current Logged Selections: {len(user_selections)} tracks registered.")
        
        if current_mode == 'recommendations':
            user_input = input("📝 Enter 'next'/'prev' to browse matches, or 'exit' to quit: ").lower().strip()
        else:
            user_input = input("📝 Enter item # to rate, 'search', 'next'/'prev', 'list', 'restart' or 'done': ").lower().strip()

        if user_input == 'search':
            IPython.display.clear_output(wait=True)
            search_results = flexible_two_input_search(master_df)

            if search_results:
                active_view_tracks = search_results
                current_mode = 'search'
                current_page = 0
            else:
                active_view_tracks = master_songs_list
                current_mode = 'browse'
                current_page = 0
            continue

        elif user_input == 'back':
            active_view_tracks = master_songs_list
            current_mode = 'browse'
            current_page = 0
            continue
        
        elif user_input == 'next':
            max_page_idx = (len(active_view_tracks) - 1) // 10
            if current_page < max_page_idx:
                current_page += 1
            continue

        elif user_input == 'prev':
            if current_page > 0:
                current_page -= 1
            continue

        elif user_input == 'list':
            if user_selections:
                print("📋 === Your Logged Preference Profile ===")
                for i, item in enumerate(user_selections):
                    print(f"[{i+1}]. {item[0]} by {item[1]} -> {item[2]}⭐")
                input("Press Enter to return to application matrix...")
            else:
                input("⚠️ Preference profile is empty! Log ratings first. Press Enter...")
            
        elif user_input == 'restart':
            user_selections = []
            active_view_tracks = master_songs_list
            current_mode = 'browse'
            current_page = 0
            print("🧹 Session memory wiped! Let's build a fresh vibe spectrum.")
            input("Press Enter to start configuration...")
            continue
                
        elif user_input.isdigit():
            target_idx = int(user_input)
            is_valid_range = False

            if current_mode == 'browse':
                start_bound = (current_page * 10) + 1
                end_bound = min((start_bound + 9), len(active_view_tracks))
                if (start_bound <= target_idx <= end_bound):
                    is_valid_range = True
            else:
                if 1 <= target_idx <= len(active_view_tracks):
                    is_valid_range = True 

            if is_valid_range: 
                target_lookup_position = target_idx - 1
                found_song = active_view_tracks[target_lookup_position]
                rating_input = input(f"⭐ Rate {found_song[0]} from 1 to 5 stars: ")

                if rating_input.isdigit() and int(rating_input) in range(1, 6):
                    for i in range(len(user_selections) - 1, -1, -1):
                        prev_song_name = user_selections[i][0]
                        prev_artist_name = user_selections[i][1]
                        if found_song[0] == prev_song_name and found_song[1] == prev_artist_name:
                            user_selections.pop(i)
                    
                    user_selections.append([found_song[0], found_song[1], int(rating_input)])
                    print(f"✅ Saved: '{found_song[0]}' -> {rating_input}★!")
                    
                    if current_mode == 'search':
                        search_results = flexible_two_input_search(master_df)
                        if search_results:
                            active_view_tracks = search_results
                            current_page = 0
                        else:
                            print("\nReturning to default onboarding layout...")
                            active_view_tracks = master_songs_list
                            current_mode = 'browse'
                            current_page = 0
                            input("Press Enter to continue...")
                else: 
                    input("❌ Invalid rating value! Choice must be 1-5. Press Enter...")
            else: 
                input("❌ Selection index sits outside view spectrum. Press Enter...")
            continue
    
        elif user_input == 'done':
            if user_selections:
                IPython.display.clear_output(wait=True)
                
                # =================================================================
                # SUB-STEP: MULTI-GEOGRAPHIC REGION SELECTION TIER
                # =================================================================
                print("🌐 --- RECOMMENDATION GEOGRAPHIC FILTER ---")
                available_regions = sorted([str(r) for r in master_df['region_1'].unique() if pd.notna(r) and r != "Global"])

                print("Filter recommendations to specific music scenes or markets.")
                print("You can select multiple options by separating them with commas (e.g., 1, 3, 5)\n")
                
                for idx, region in enumerate(available_regions, start=1):
                    print(f"[{idx}] {region}")
                global_option_idx = len(available_regions) + 1
                print(f"[{global_option_idx}] No Region Filter (Explore Open Global Catalog)")

                raw_geo_input = input(f"\nSelect option(s) (1 to {global_option_idx}): ").strip()

                input_choices = [c.strip() for c in raw_geo_input.split(",") if c.strip()]

                target_regions = []
                for choice in input_choices:
                    if choice.isdigit():
                        ch_idx = int(choice) - 1
                        if 0 <= ch_idx < len(available_regions):
                            target_regions.append(available_regions[ch_idx])
                        elif int(choice) == global_option_idx:
                            target_regions = []
                            break
                
                if not target_regions:
                    target_regions = None
                    print("🌍 Exploring the open Global catalog...")
                else:
                    print(f"✅ Geographic constraints locked to: {', '.join(target_regions)}")

                print("\n--------------------------------------------------------")
                
                # =================================================================
                # EXISTING STEP: HISTORICAL DISCOVERY FILTER
                # =================================================================             
                print("🎯 --- RECOMMENDATION FILTER CONFIGURATION ---")
                print("Do you want your final recommendations to include songs from your history database?")
                print("[1] Yes - merge history profiles together for full visibility.")
                print("[2] No - discovery mode only. Filter out previously rated/seen songs.")

                filter_choice = input("Select option (1 or 2): ").strip()

                exclude_set = []
                if filter_choice == "2":
                    historical_pool = load_interacted_songs_from_disk(history_file_path)
                    for item in historical_pool:
                        exclude_set.append([item[0], item[1]])
                    for item in user_selections:
                        exclude_set.append([item[0], item[1]])

                print("⚙️ Intersecting preference metrics against vector repositories...") 
                taste_vector = calculate_taste_vector(user_selections, master_df)
                
                # Hand data over safely to the vector engine
                recommendation_pool = generate_geo_recommendations(
                    test_vector=taste_vector, 
                    master_df=master_df, 
                    exclude_tracks=exclude_set, 
                    target_region=target_regions
                )
                active_view_tracks = recommendation_pool
                current_mode = 'recommendations'
                current_page = 0
            else:
                print("Session terminated with empty preference profile. Goodbye!")
                break
            continue
            
        elif user_input == 'exit' and current_mode == 'recommendations':
            save_interacted_songs_to_disk(user_selections, history_file_path)
            print("💾 Session history archived to disk. Thanks for listening!")
            break

# ==================================================================================================

In [8]:
if __name__ == "__main__":
    print("💿 Loading 1.08M track catalog dataset into memory...")
    
    if os.path.exists("tracks_features.parquet"):
        df = pd.read_parquet("tracks_features.parquet", engine="pyarrow")
    else:
        print("⚠️ Parquet file not found. Falling back to slow CSV load...")
        df = pd.read_csv('tracks_features.csv')
        df = df[(df['duration_ms'] >= 60000) & (df['duration_ms'] <= 600000)]
        df = df.drop_duplicates(subset=['name', 'artists'])
        df['clean_artists'] = df['artists'].str.replace(r"[\[\]']", "", regex=True)
        df['search_text'] = df['name'] + " - " + df['clean_artists']
        df['name_prefix'] = df['name'].str.lower().str[:2]
        df['artist_prefix'] = df['clean_artists'].str.lower().str[:2]

    print("Advanced Search & Pagination Systems Active!")

    history_file = "user_interaction_history.json"
    starting_tracks = initialize_onboarding_tracks(df, history_file)
    run_interaction_loop(starting_tracks, df, history_file)

✨ --- YOUR PERSONALIZED RECOMMENDATIONS SPECTRUM ---

 --- SONG LIST 1 (Songs 1 to 10) ---
[1] When the Crazy Kicks In by Francesca Battistelli
[2] Melancholy Kaleidoscope by All Time Low
[3] Headstrong by Trapt
[4] Temptation by Trik Turner
[5] Picture Of You by Elwood
[6] Spring Released by Grant-Lee Phillips
[7] Who We Are by 3PM
[8] On My Own by Shamir
[9] Apple Face (Blue Skies) by Royal Peeps
[10] Thread by The Sonic Revolution
----------------------------------------------
🔘 [ ⬅️ Previous 10 ]  |  🔘 [ ➡️ Next 10 ]  |  🔘 [ Done ✅ ]
--------------------------------------------------------
Options: ['next'/'prev' to flip pages] | ['exit' to close application]
🛒 Current Logged Selections: 1 tracks registered.
💾 Session history archived to disk. Thanks for listening!


-> request and clean title input

-> if title tnput is `#escape` then return `[]`

-> request and clean artist input

-> if artist tnput is `#escape` then return `[]`

-> if not title input and artist input then return `[]`

-> matched df = master df

-> artits_prefix = artist_input[:1]
-> title_prefix = title_input[:1]

-> possible artists = [
    [artist for artist in df['clean_artists'] if artist.contains(artist_input)],
    [artist for artist in df['clean_artists'] if artist.startswith(artist_prefix)],
]

-> possible titles = [
    [artist for artist in df['clean_artists'] if artist.contains(artist_input)],
    [artist for artist in df['clean_artists'] if artist.startswith(artist_prefix)],
]

-> if matched_df is empty --> (probably wont be, but need to think of an endpoint)

-> best_match_title = fuzz(title input, matched_df[titles])

-> best_match_artits = fuzz(artist input, matched_df[artists])

-> matched df = matched_df [
    (matched_df[title] == best_match_title) |
    (matched_df[artist] == best_match_artist)
]

-> lookup_key = [title - artist]
-> best match combined = (lookup key, input)

return the table

In [ ]:

# ==================================================================================================
# === 2. FLEXIBLE PREFIX-FUZZY SEARCH ENGINE =======================================================
# ==================================================================================================
def flexible_two_input_search(master_df) -> list:
    """
    Filters the catalog using a 2-character title prefix slice before passing 
    candidates to a fuzzy string matcher to rank search relevance.
    """
    print("\n🔍 --- NEW MATRIX SEARCH ---")
    print("(Press Enter on both fields or type 'escape' to return to browsing mode)")
    title_input = input("🎵 Enter Song Title (or press Enter to skip): ").strip().lower()
    
    if title_input == '#escape':
        return []
        
    artist_input = input("👤 Enter Artist Name (or press Enter to skip): ").strip().lower()
    
    if artist_input == 'escape':
        return []

    if not title_input and not artist_input:
        return []
    
    matched_df = master_df
    if title_input:
        title_prefix = title_input[:1]
        possible_titles = [
            [name for name in matched_df['name'] if (name[:1] == title_prefix) | (title_input in name.split())]
        ]
        
        if not prefix_matches.empty:
            matched_df = prefix_matches
            if artist_input:
                artist_subset = matched_df[matched_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]
                if not artist_subset.empty:
                    matched_df = artist_subset
        else:
            if artist_input:
                matched_df = master_df[master_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]
    elif artist_input:
        matched_df = master_df[master_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]

    if matched_df.empty:
        print("⚠️ No exact filters found. Scanning global text pool...")
        matched_df = master_df
    
    lookup_dictionary = {}
    for row in matched_df.itertuples():
        lookup_key = f"{str(row.name).lower()} - {str(row.clean_artists).lower()}"
        lookup_dictionary[lookup_key] = [row.name, row.clean_artists]

    choices_list = list(lookup_dictionary.keys())

    if title_input and artist_input:
        target_string = f"{title_input} - {artist_input}"
    elif title_input:
        target_string = title_input
    else: 
        target_string = artist_input

    raw_matches = process.extract(target_string, choices_list, limit=None)

    formatted_matches = []
    match_counter = 1
    print("\n 🎯 Matches Found:")

    for match_item, confidence_score in raw_matches:
        if confidence_score >= 70:
            matches = lookup_dictionary[match_item]
            formatted_matches.append(matches)
            print(f"[{match_counter}] {matches[0]} by {matches[1]} (Confidence: {int(confidence_score)})")
            match_counter += 1

    return formatted_matches



In [ ]:

# ==================================================================================================
# === 2. FLEXIBLE PREFIX-FUZZY SEARCH ENGINE =======================================================
# ==================================================================================================
def two_input_search(master_df) -> list:
    """
    Filters the catalog using a 2-character title prefix slice before passing 
    candidates to a fuzzy string matcher to rank search relevance.
    """
    print("\n🔍 --- NEW MATRIX SEARCH ---")
    print("(Press Enter on both fields or type 'escape' to return to browsing mode)")
    title_input = input("🎵 Enter Song Title (or press Enter to skip): ").strip().lower()
    
    if title_input == '#escape':
        return []
        
    artist_input = input("👤 Enter Artist Name (or press Enter to skip): ").strip().lower()
    
    if artist_input == 'escape':
        return []

    if not title_input and not artist_input:
        return []
    
    matched_df = master_df
    if title_input:
        title_prefix = title_input[:2]
        prefix_matches = master_df[master_df['name_prefix'] == title_prefix]
        
        if not prefix_matches.empty:
            matched_df = prefix_matches
            if artist_input:
                artist_subset = matched_df[matched_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]
                if not artist_subset.empty:
                    matched_df = artist_subset
        else:
            if artist_input:
                matched_df = master_df[master_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]
    elif artist_input:
        matched_df = master_df[master_df['clean_artists'].str.lower().str.contains(artist_input, na=False)]

    if matched_df.empty:
        print("⚠️ No exact filters found. Scanning global text pool...")
        matched_df = master_df
    
    lookup_dictionary = {}
    for row in matched_df.itertuples():
        lookup_key = f"{str(row.name).lower()} - {str(row.clean_artists).lower()}"
        lookup_dictionary[lookup_key] = [row.name, row.clean_artists]

    choices_list = list(lookup_dictionary.keys())

    if title_input and artist_input:
        target_string = f"{title_input} - {artist_input}"
    elif title_input:
        target_string = title_input
    else: 
        target_string = artist_input

    raw_matches = process.extract(target_string, choices_list, limit=None)

    formatted_matches = []
    match_counter = 1
    print("\n 🎯 Matches Found:")

    for match_item, confidence_score in raw_matches:
        if confidence_score >= 70:
            matches = lookup_dictionary[match_item]
            formatted_matches.append(matches)
            print(f"[{match_counter}] {matches[0]} by {matches[1]} (Confidence: {int(confidence_score)})")
            match_counter += 1

    return formatted_matches



In [6]:
flexible_two_input_search(df)


🔍 --- NEW MATRIX SEARCH ---
(Press Enter on both fields or type 'escape' to return to browsing mode)

 🎯 Matches Found:
[1] Adore - Instrumental by Nicolay (Confidence: 86)
[2] Adore You Parody by Bart Baker (Confidence: 86)
[3] Adore (Fall in Love Forever) by William Control (Confidence: 86)
[4] Adore - Acoustic Version by Jasmine Thompson (Confidence: 86)
[5] Adore (I Love You) by Norman Connors (Confidence: 86)
[6] Adore: O Come All Ye Faithful / What Child Is This / Away in a Manger / The First Noel / O Holy Night by His Own (Confidence: 86)
[7] Adore You (feat. Abhi Dijon) by Nao, Abhi//Dijon (Confidence: 86)
[8] Adore by Seven (Confidence: 81)


[['Adore - Instrumental', 'Nicolay'],
 ['Adore You Parody', 'Bart Baker'],
 ['Adore (Fall in Love Forever)', 'William Control'],
 ['Adore - Acoustic Version', 'Jasmine Thompson'],
 ['Adore (I Love You)', 'Norman Connors'],
 ['Adore: O Come All Ye Faithful / What Child Is This / Away in a Manger / The First Noel / O Holy Night',
  'His Own'],
 ['Adore You (feat. Abhi Dijon)', 'Nao, Abhi//Dijon'],
 ['Adore', 'Seven']]